In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device()#drop=['cuda:1','cuda:6'])

#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
# _batch = d.get_toy_databatch(dataset, device)

('cuda:1', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 79361)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 74623)

# #### Device() ####
# device = cuda:1

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:1)
# edge_attr                (32464, 16)              Tensor (cuda:1)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:1)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

In [2]:
from modules.layers import MultiheadSetPooling
from modules.model import LatentClassifier
from modules.norm import ZlogNorm
from modules.train import ClassifTrainer, Experiment, grid

import torch.nn as nn
from functools import partial

initial multihead: 64/128 embed dim, 2 heads, concat

In [3]:
model_template = partial(
    LatentClassifier,
    dataset=dataset,
    hidden_dims = 2,
    method='set',
    
    # layers
    norm_class=ZlogNorm,
    encoder_class=nn.Linear,
    pooling_class=MultiheadSetPooling,
    mlp=False,

    # layer params
    act_fn=nn.ReLU, 
    norm_fn='layer', 
    end_fn=False,
)

model_grid = grid(
    model_template,
    embed_dim = [64,128],
    num_heads = [1,2,4,8],
    pooling_kwargs= [{'aggregate':'concat'}, {'aggregate':'mean'}]
)

trainer_template = partial(
    ClassifTrainer,
    loss_class=nn.CrossEntropyLoss,
    lr=5e-4,
    report_metrics=['loss','accuracy'],
    weight_method='balanced',
)

trainer_grid = grid(
    trainer_template,
)

expt = Experiment(
    num_trials=10,
    num_epochs=50,
    dataset=dataset,
    device=device,
    batch_size=128
)

expt.add_grid(
    model_grid=model_grid,
    trainer_grid=trainer_grid
)

print(len(expt.configs))

16


In [4]:
expt.run_experiment(
    'classif_grid4b_multihead',
    save_csv=True,
    save_params=True,
    save_values=True,
    verbose=False,
)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

checking if interaction with mlp + multihead method

In [5]:
# model_template = partial(
#     LatentClassifier,
#     dataset=dataset,
#     embed_dim = 64,
#     hidden_dims = 2,
#     method='set',
    
#     # layers
#     norm_class=ZlogNorm,
#     encoder_class=nn.Linear,
#     pooling_class=MultiheadSetPooling,

#     # layer params
#     act_fn=nn.ReLU, 
#     norm_fn='layer', 
#     end_fn=False,
# )

# model_grid = grid(
#     model_template,
#     mlp=[False,True],
#     num_heads = [1,4],
#     pooling_kwargs= [{'aggregate':'concat'}, {'aggregate':'mean'}]
# )

# trainer_template = partial(
#     ClassifTrainer,
#     loss_class=nn.CrossEntropyLoss,
#     lr=5e-4,
#     report_metrics=['loss','accuracy'],
#     weight_method='balanced',
# )

# trainer_grid = grid(
#     trainer_template,
# )

# expt = Experiment(
#     num_trials=10,
#     num_epochs=50,
#     dataset=dataset,
#     device=device,
#     batch_size=128
# )

# expt.add_grid(
#     model_grid=model_grid,
#     trainer_grid=trainer_grid
# )

# print(len(expt.configs))

In [6]:
# expt.run_experiment(
#     'classif_grid4b_multihead_mlp',
#     save_csv=True,
#     save_params=True,
#     save_values=True,
#     verbose=False,
# )